# Lesson 14b: Generative Models — Practical

14a derived the VAE's evidence lower bound and diffusion's closed-form
forward process by hand, in NumPy, on toy-scale problems. This notebook
trains a real VAE and a real (small) diffusion model on MNIST with PyTorch,
and compares what each family's derived objective actually buys in
practice: sample quality and training stability.

## Introduction

Both models trained below optimise the objectives 14a derived — the VAE
maximises the ELBO (reconstruction minus KL), the diffusion model minimises
the simplified noise-prediction loss $L_{\text{simple}}$ — but at a scale
where the difference between "a formula on a whiteboard" and "an actual
generative model" becomes visible: does sampling $z \sim \mathcal N(0, I)$
and decoding really produce a digit now that the KL term has done its job,
and does the diffusion model's many-small-steps design actually turn pure
noise into a recognisable image?

## Setup

In [ ]:
# Fixed seeds: every stochastic step in this notebook (weight init, minibatch
# order, noise sampling) is reproducible.
import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import torchvision
from torchvision import transforms
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (5, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

# Pinned to CPU rather than the usual "cuda if available" check: this
# notebook is gated on CPU (under 10 minutes, no GPU ever required, per
# every notebook in this series), and every model here is small enough
# that CPU training costs seconds, not minutes.
device = torch.device("cpu")
print("using device:", device)

In [ ]:
mnist = torchvision.datasets.MNIST(root="data", train=True, download=True,
                                    transform=transforms.ToTensor())

N = 3000
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(mnist))[:N]
X01 = torch.stack([mnist[i][0].reshape(-1) for i in idx])  # (N, 784), pixels in [0, 1]

fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for ax, i in zip(axes, range(8)):
    ax.imshow(X01[i].reshape(28, 28), cmap="gray")
    ax.axis("off")
plt.suptitle("MNIST training samples")
plt.show()
print("X01:", X01.shape)

## Training a VAE

The encoder outputs $(\mu, \log\sigma^2)$ for a diagonal Gaussian
$q_\phi(z\mid x)$; the decoder maps a sampled $z$ back to pixel-space
probabilities. The loss is 14a's ELBO, negated (since PyTorch's optimisers
minimise): binary cross-entropy reconstruction plus the closed-form Gaussian
KL, both derived in 14a — and $z$ is drawn via the reparameterisation trick
so gradients reach the encoder.

In [ ]:
Z_DIM = 20

class VAE(nn.Module):
    def __init__(self, d=784, h=256, z_dim=Z_DIM):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d, h), nn.ReLU())
        self.to_mu = nn.Linear(h, z_dim)
        self.to_logvar = nn.Linear(h, z_dim)
        self.decoder = nn.Sequential(nn.Linear(z_dim, h), nn.ReLU(), nn.Linear(h, d), nn.Sigmoid())

    def encode(self, x):
        h = self.encoder(x)
        return self.to_mu(h), self.to_logvar(h)

    def reparameterise(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)      # 14a's reparameterisation trick: z = mu + std * eps
        return mu + std * eps

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        return self.decoder(z), mu, logvar


def vae_loss(xhat, x, mu, logvar):
    recon = nn.functional.binary_cross_entropy(xhat, x, reduction="sum") / x.shape[0]
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.shape[0]  # 14a's closed-form Gaussian KL
    return recon + kl, recon, kl


torch.manual_seed(SEED)
vae = VAE().to(device)
vae_opt = optim.Adam(vae.parameters(), lr=1e-3)

BATCH_SIZE = 128
vae_history = {"loss": [], "recon": [], "kl": []}
for epoch in range(60):
    perm = torch.randperm(N)
    epoch_losses, epoch_recons, epoch_kls = [], [], []
    for s in range(0, N, BATCH_SIZE):
        xb = X01[perm[s:s + BATCH_SIZE]]
        xhat, mu, logvar = vae(xb)
        loss, recon, kl = vae_loss(xhat, xb, mu, logvar)
        vae_opt.zero_grad(); loss.backward(); vae_opt.step()
        epoch_losses.append(loss.item()); epoch_recons.append(recon.item()); epoch_kls.append(kl.item())
    vae_history["loss"].append(np.mean(epoch_losses))
    vae_history["recon"].append(np.mean(epoch_recons))
    vae_history["kl"].append(np.mean(epoch_kls))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(vae_history["loss"], label="total (-ELBO)")
ax.plot(vae_history["recon"], label="reconstruction")
ax.plot(vae_history["kl"], label="KL")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend(); ax.set_title("VAE training")
plt.show()
print(f"final loss: {vae_history['loss'][-1]:.2f} (recon {vae_history['recon'][-1]:.2f} + KL {vae_history['kl'][-1]:.2f})")

In [ ]:
vae.eval()
with torch.no_grad():
    xhat, _, _ = vae(X01[:8])

fig, axes = plt.subplots(2, 8, figsize=(13, 3.5))
for i in range(8):
    axes[0, i].imshow(X01[i].reshape(28, 28), cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(xhat[i].reshape(28, 28), cmap="gray"); axes[1, i].axis("off")
axes[0, 0].set_ylabel("original", fontsize=9)
axes[1, 0].set_ylabel("reconstruction", fontsize=9)
plt.suptitle("VAE reconstructions")
plt.show()

## Sampling from the Prior

14a's untrained, unregularised autoencoder decoded random bottleneck codes
into noise. The whole point of the KL term is that a *trained* VAE's
encoder output distribution is pulled toward $\mathcal N(0, I)$ specifically
so that sampling $z$ directly from that same prior — no encoder, no real
input image involved at all — decodes into something recognisable.

In [ ]:
torch.manual_seed(SEED + 1)
with torch.no_grad():
    z_prior = torch.randn(8, Z_DIM)          # sampled directly from p(z) = N(0, I), no input image
    prior_samples = vae.decoder(z_prior)

fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for ax, i in zip(axes, range(8)):
    ax.imshow(prior_samples[i].reshape(28, 28), cmap="gray")
    ax.axis("off")
plt.suptitle("VAE: decoding z ~ N(0, I) directly from the prior (no encoder involved)")
plt.show()

These are recognisably digit-like — blurrier than the reconstructions above,
but structurally nothing like 14a's random-code noise. The KL term is the
entire reason: it is what makes $z \sim \mathcal N(0, I)$ land somewhere the
decoder has actually learned to interpret.

## A Minimal Diffusion Model

A full image-diffusion U-Net is far outside this notebook's CPU budget, but
the mechanism 14a derived does not require one: a small convolutional network
that takes a noisy image $x_t$ and a timestep $t$ and predicts the noise
$\epsilon$ added to it is a legitimate, if minimal, instance of the same
$\epsilon_\theta(x_t, t)$ 14a's $L_{\text{simple}}$ objective calls for. A flat
MLP was tried first and measurably failed at this: its eps-prediction loss
plateaued far above zero and its reverse samples never left pure noise,
because a plain MLP has no spatial inductive bias for pixel-grid structure.
A handful of full-resolution convolutional layers (no downsampling — the
image is already tiny) fixes this cheaply. Pixels are rescaled to
$[-1, 1]$ (diffusion's usual convention, matching the $\mathcal N(0, I)$
prior the forward process converges to) rather than kept in $[0, 1]$.

In [ ]:
X = X01 * 2 - 1  # rescale to [-1, 1] for diffusion

T = 1000  # standard DDPM schedule: same beta range as before, but T=1000 so alpha_bar_T is
# genuinely near zero -- matches 14a and closes the train/inference mismatch with
# the N(0, I) start sample_diffusion below uses
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)


def sinusoidal_embedding(t, dim):
    # standard transformer-style timestep embedding: unlike a lookup table,
    # nearby timesteps get similar vectors, so the network can generalise
    # across t instead of learning each of the T indices independently
    half = dim // 2
    freqs = torch.exp(-np.log(10000) * torch.arange(half, dtype=torch.float32) / half)
    args = t[:, None].float() * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class ConvBlock(nn.Module):
    # full-resolution (no downsampling -- 28x28 is already tiny): each block
    # is one 3x3 conv plus a per-channel bias projected from the timestep
    def __init__(self, c_in, c_out, t_embed_dim):
        super().__init__()
        self.conv = nn.Conv2d(c_in, c_out, kernel_size=3, padding=1)
        self.t_proj = nn.Linear(t_embed_dim, c_out)
        self.act = nn.ReLU()

    def forward(self, x, t_emb):
        h = self.conv(x) + self.t_proj(t_emb)[:, :, None, None]
        return self.act(h)


class DenoiseCNN(nn.Module):
    def __init__(self, channels=32, t_embed_dim=32):
        super().__init__()
        self.t_embed_dim = t_embed_dim
        self.t_mlp = nn.Sequential(nn.Linear(t_embed_dim, t_embed_dim), nn.ReLU())
        self.block1 = ConvBlock(1, channels, t_embed_dim)
        self.block2 = ConvBlock(channels, channels, t_embed_dim)
        self.block3 = ConvBlock(channels, channels, t_embed_dim)
        self.out = nn.Conv2d(channels, 1, kernel_size=3, padding=1)

    def forward(self, x_flat, t):
        x = x_flat.view(-1, 1, 28, 28)
        t_emb = self.t_mlp(sinusoidal_embedding(t, self.t_embed_dim))
        h = self.block1(x, t_emb)
        h = self.block2(h, t_emb)
        h = self.block3(h, t_emb)
        return self.out(h).view(-1, 784)


torch.manual_seed(SEED)
denoiser = DenoiseCNN().to(device)
diff_opt = optim.Adam(denoiser.parameters(), lr=1e-3)

diff_history = []
for epoch in range(100):
    perm = torch.randperm(N)
    epoch_losses = []
    for s in range(0, N, BATCH_SIZE):
        xb = X[perm[s:s + BATCH_SIZE]]
        n = xb.shape[0]
        t = torch.randint(0, T, (n,))
        eps = torch.randn_like(xb)
        ab = alpha_bars[t].unsqueeze(1)
        xt = torch.sqrt(ab) * xb + torch.sqrt(1 - ab) * eps   # 14a's closed-form forward process
        eps_pred = denoiser(xt, t)
        loss = nn.functional.mse_loss(eps_pred, eps)          # 14a's L_simple
        diff_opt.zero_grad(); loss.backward(); diff_opt.step()
        epoch_losses.append(loss.item())
    diff_history.append(np.mean(epoch_losses))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(diff_history)
ax.set_xlabel("epoch"); ax.set_ylabel("L_simple (noise-prediction MSE)"); ax.set_title("Diffusion training")
plt.show()
print(f"final L_simple: {diff_history[-1]:.4f}")

Sampling runs the reverse process 14a described: start from pure noise
$x_T \sim \mathcal N(0, I)$ and repeatedly subtract the model's predicted
noise, one small step at a time, down to $x_0$.

In [ ]:
@torch.no_grad()
def sample_diffusion(model, n_samples, seed, record_every=200):
    torch.manual_seed(seed)
    x = torch.randn(n_samples, 784)
    snapshots = {T: x.clone()}
    for t_step in reversed(range(T)):
        t = torch.full((n_samples,), t_step, dtype=torch.long)
        eps_pred = model(x, t)
        alpha_t, alpha_bar_t, beta_t = alphas[t_step], alpha_bars[t_step], betas[t_step]
        mean = (x - (beta_t / torch.sqrt(1 - alpha_bar_t)) * eps_pred) / torch.sqrt(alpha_t)
        x = mean + torch.sqrt(beta_t) * torch.randn_like(x) if t_step > 0 else mean
        if t_step % record_every == 0:
            snapshots[t_step] = x.clone()
    return x, snapshots


final_samples, snapshots = sample_diffusion(denoiser, n_samples=8, seed=SEED)

recorded_ts = sorted(snapshots.keys(), reverse=True)
fig, axes = plt.subplots(len(recorded_ts), 8, figsize=(13, 1.6 * len(recorded_ts)))
for row, t_step in enumerate(recorded_ts):
    imgs = (snapshots[t_step].clamp(-1, 1) + 1) / 2  # back to [0, 1] for display
    for col in range(8):
        axes[row, col].imshow(imgs[col].reshape(28, 28), cmap="gray")
        axes[row, col].axis("off")
    axes[row, 0].set_title(f"t={t_step}", loc="left", fontsize=9)
plt.suptitle("Reverse process: pure noise (top) to generated digits (bottom)")
plt.tight_layout(); plt.show()

## Comparison

Sample quality, side by side, and the two training curves together —
quality traded against what each objective actually is.

In [ ]:
torch.manual_seed(SEED + 2)
with torch.no_grad():
    vae_final_samples = vae.decoder(torch.randn(8, Z_DIM))
diffusion_final_samples = (final_samples.clamp(-1, 1) + 1) / 2

fig, axes = plt.subplots(2, 8, figsize=(13, 3.5))
for i in range(8):
    axes[0, i].imshow(vae_final_samples[i].reshape(28, 28), cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(diffusion_final_samples[i].reshape(28, 28), cmap="gray"); axes[1, i].axis("off")
plt.suptitle("VAE prior samples (top) vs. diffusion reverse-process samples (bottom)")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(vae_history["loss"]); axes[0].set_title("VAE: -ELBO per epoch"); axes[0].set_xlabel("epoch")
axes[1].plot(diff_history); axes[1].set_title("Diffusion: L_simple per epoch"); axes[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

Both training curves fall smoothly and monotonically — at this scale, on
this data, neither objective shows the other's textbook failure mode (a
GAN's loss, by contrast, oscillates by design, since the generator and
discriminator are chasing each other rather than jointly minimising a
shared quantity). The VAE's prior samples are recognisably digit-shaped,
if blurry. The diffusion model's reverse-sampled images show real
stroke-and-curve structure — a clear step up from pure noise, and visible
proof the reverse process is doing something — but they are not yet
crisp, fully-formed digits the way the VAE's are. $L_{\text{simple}}$ is
far lower here than an earlier flat-MLP denoiser reached (that version
plateaued around 0.65-0.93 and its reverse samples never left pure
noise); the fix was replacing the MLP with a small full-resolution
convolutional network, giving the model the spatial inductive bias
pixel-grid denoising needs. What remains is a capacity ceiling, not a
missing mechanism: 23K parameters and 100 epochs is a fraction of what a
real diffusion U-Net uses, and sharper samples would need more of both,
trading past this notebook's CPU budget.

## Key Takeaways

- A VAE trained with 14a's ELBO objective closes the exact gap 14a's
  untrained-autoencoder experiment exposed: sampling $z \sim \mathcal N(0, I)$
  directly from the prior — no input image, no encoder — decodes into a
  recognisable digit, because the KL term specifically pulled the encoder's
  output distribution to match that prior during training.
- A minimal convolutional diffusion model, trained on 14a's
  $L_{\text{simple}}$ noise-prediction loss, reverses pure Gaussian noise
  into images with genuine digit-like stroke and curve structure over
  1000 denoising steps — a qualitative jump from a flat MLP of similar
  size, which plateaus at a far higher loss and never leaves pure noise.
  The difference is architectural: convolution gives the network the
  spatial locality and weight-sharing that pixel-grid denoising needs,
  which a fully-connected layer has no way to encode.
- **Sample quality**: the VAE's single-step decode produces recognisable,
  if blurry, digits; this minimal CNN-based diffusion model produces
  visible stroke structure but not yet crisp digits — a capacity
  ceiling (23K parameters, 100 epochs) rather than a flaw in the
  objective or the mechanism, which both notebooks derived and verified
  independently of model size.
- **Training stability**: both objectives descended smoothly here — neither
  needs the adversarial balancing a GAN's minimax game requires, which is
  precisely 14a's point about optimising a bound versus optimising a game.